# function_basics
**Prerequisites:** variables, data_types, operators, control_flow, loops

**Outcome:** After this notebook you will know exactly what a function is at the memory level, how to define and call one, what happens when Python executes a function call, and why functions are the single most important tool for writing maintainable code.

## The Problem

In [ ]:
# validating records without a function — repeated logic, impossible to maintain
record_1 = {"id": "job_1", "status": "pending", "retries": 2}
record_2 = {"id": "job_2", "status": "",        "retries": 5}
record_3 = {"id": "",      "status": "done",    "retries": 0}

if record_1["id"] and record_1["status"] and record_1["retries"] <= 3:
    print(f"{record_1['id']} is valid")
else:
    print(f"{record_1['id'] or 'unknown'} is invalid")

if record_2["id"] and record_2["status"] and record_2["retries"] <= 3:
    print(f"{record_2['id']} is valid")
else:
    print(f"{record_2['id'] or 'unknown'} is invalid")

if record_3["id"] and record_3["status"] and record_3["retries"] <= 3:
    print(f"{record_3['id']} is valid")
else:
    print(f"{record_3['id'] or 'unknown'} is invalid")

# the validation rule is written 3 times
# change the rule once and you must find and update every copy

## The Concept

A function is a named, reusable block of code that performs one task. You define it once with `def` and call it as many times as needed. Every call gets its own isolated workspace — its own local variables that disappear when the call finishes. Functions eliminate repetition, give behaviour a name, and make code testable in isolation. In Python, functions are first-class objects: they can be assigned to variables, passed as arguments, and returned from other functions.

## Minimal Example

In [ ]:
def greet(name):
    message = "Hello, " + name
    return message

result = greet("alice")
print(result)

## How Python Does It

When Python executes a `def` statement it creates a function object and binds it to the name you gave it — exactly like `name = value` creates a variable. The function object stores the code, the parameter names, the default values, and the docstring. When you call the function Python creates a new stack frame for that call, binds the arguments to the parameter names inside that frame, executes the body, and then destroys the frame when the function returns. The local variables inside the function do not exist before the call and cease to exist after it.

In [ ]:
def validate(record):
    return bool(record["id"] and record["status"])

# a function is an object — it has a type, an id, and attributes
print(type(validate))
print(id(validate))
print(validate.__name__)
print(validate.__code__.co_varnames)   # local variable names

## Building Up

In [ ]:
# function with no parameters and no return value
def print_separator():
    print("-" * 40)

print_separator()
print("section one")
print_separator()
print("section two")
print_separator()

In [ ]:
# function with parameters and a return value
def is_valid_record(record):
    if not record.get("id"):
        return False
    if not record.get("status"):
        return False
    if record.get("retries", 0) > 3:
        return False
    return True

records = [
    {"id": "job_1", "status": "pending", "retries": 2},
    {"id": "job_2", "status": "",        "retries": 5},
    {"id": "",      "status": "done",    "retries": 0},
]

for record in records:
    label = "valid" if is_valid_record(record) else "invalid"
    print(f"{record['id'] or 'no-id'}: {label}")

In [ ]:
# docstring — documents what a function does, not how
def compute_retry_delay(attempt, base=2):
    """
    Returns the delay in seconds before the next retry attempt.
    Uses exponential backoff: base ** attempt.

    Args:
        attempt: the retry attempt number, starting from 0
        base: the base for exponential calculation, default 2

    Returns:
        int: delay in seconds
    """
    return base ** attempt

for i in range(5):
    print(f"attempt {i}: wait {compute_retry_delay(i)}s")

In [ ]:
# functions are objects — assignable to variables
def tag_record(record):
    return {**record, "tagged": True}

process = tag_record          # no () — assign the function object, not call it
print(process({"id": "job_1", "status": "pending"}))

In [ ]:
# early return — exit as soon as the answer is known
def find_failed(jobs):
    for job in jobs:
        if job["status"] == "failed":
            return job           # return immediately — no need to scan the rest
    return None                  # none found

jobs = [
    {"id": "job_1", "status": "done"},
    {"id": "job_2", "status": "failed"},
    {"id": "job_3", "status": "pending"},
]

failed = find_failed(jobs)
print(failed)

In [ ]:
# a function with no explicit return returns None
def log_event(event):
    print(f"[LOG] {event}")

result = log_event("boot")
print(result)           # None
print(type(result))     # <class 'NoneType'>

## Where It Breaks

In [ ]:
# calling a function before defining it
try:
    result = add(1, 2)    # NameError — add does not exist yet
except NameError as e:
    print(e)

def add(a, b):
    return a + b

print(add(1, 2))          # works now — def has executed

In [ ]:
# forgetting () calls the function — omitting () references the object
def get_status():
    return "running"

print(get_status)    # <function get_status at 0x...> — the function object
print(get_status())  # "running" — the return value

## The Fix

In [ ]:
# the problem from the top — solved with a function
def is_valid_record(record):
    return bool(
        record.get("id")
        and record.get("status")
        and record.get("retries", 0) <= 3
    )

records = [
    {"id": "job_1", "status": "pending", "retries": 2},
    {"id": "job_2", "status": "",        "retries": 5},
    {"id": "",      "status": "done",    "retries": 0},
]

for record in records:
    label = "valid" if is_valid_record(record) else "invalid"
    print(f"{record['id'] or 'no-id'}: {label}")

# change the rule once — all three checks update automatically

## In a Real System

In [ ]:
# a data pipeline broken into named functions — each does one thing
def load_records(source):
    return [{"id": f"job_{i}", "value": i * 10, "status": "raw"}
            for i in range(len(source))]

def filter_valid(records):
    return [r for r in records if r["value"] > 0]

def enrich(records):
    return [{**r, "status": "enriched"} for r in records]

def summarise(records):
    total = sum(r["value"] for r in records)
    return {"count": len(records), "total": total}

source    = ["a", "b", "c", "d"]
raw       = load_records(source)
valid     = filter_valid(raw)
enriched  = enrich(valid)
summary   = summarise(enriched)

print(summary)

## Performance

Every function call in Python creates a new stack frame — there is a small but real overhead compared to inlining the same code. For most code this is irrelevant. In tight inner loops called millions of times per second, the call overhead can add up. In those cases inline the logic or use built-in functions which are implemented in C and have no Python frame overhead. Never optimise away named functions for readability — profile first.

## Anti-Pattern

In [ ]:
# anti-pattern: one function that does too many things
def process(records, output_file, max_retries, region, dry_run):
    valid = [r for r in records if r.get("id") and r.get("status")]
    enriched = [{**r, "region": region} for r in valid]
    if not dry_run:
        with open(output_file, "w") as f:
            for r in enriched:
                f.write(str(r) + "\n")
    return enriched
# loads, validates, enriches, and writes — untestable in isolation

# correct: one function, one responsibility
def filter_valid(records):
    return [r for r in records if r.get("id") and r.get("status")]

def enrich_with_region(records, region):
    return [{**r, "region": region} for r in records]

def write_records(records, output_file):
    with open(output_file, "w") as f:
        for r in records:
            f.write(str(r) + "\n")

# each can be tested, replaced, and reasoned about independently

## Interview Signals

- What happens in memory when Python executes a `def` statement?
- What does a function return if it has no explicit `return` statement?
- What is the difference between referencing a function and calling it?
- Why are functions described as first-class objects in Python?

## Exercise

In [ ]:
def summarise_jobs(jobs):
    """
    Takes a list of job dicts, each with keys 'id', 'status', 'duration_ms'.
    Returns a dict with:
        - 'total'   : total number of jobs
        - 'done'    : count of jobs where status == 'done'
        - 'failed'  : count of jobs where status == 'failed'
        - 'avg_ms'  : average duration_ms across all jobs, rounded to 2 decimals
                      return 0 if jobs list is empty

    Bug: the current implementation is incomplete.
    Fill in the missing logic so all assertions pass.
    """
    return {}


jobs = [
    {"id": "job_1", "status": "done",   "duration_ms": 100},
    {"id": "job_2", "status": "failed", "duration_ms": 200},
    {"id": "job_3", "status": "done",   "duration_ms": 150},
    {"id": "job_4", "status": "pending","duration_ms": 50},
]

result = summarise_jobs(jobs)

assert result["total"]  == 4,      f"got {result.get('total')}"
assert result["done"]   == 2,      f"got {result.get('done')}"
assert result["failed"] == 1,      f"got {result.get('failed')}"
assert result["avg_ms"] == 125.0,  f"got {result.get('avg_ms')}"
assert summarise_jobs([]) == {"total": 0, "done": 0, "failed": 0, "avg_ms": 0}
print("all assertions passed")